# Fine-tuning Voxtral Realtime on Colab

Runs the fine-tuning (`docs/cloud.md`) with one script. This notebook only drives
`scripts/run_finetune.py` and holds no logic of its own.

**Before you start**
1. *Runtime → Change runtime type → GPU: **A100**.* (L4 works too, slower; not the T4.)

Run the cells in order. Everything is saved to Google Drive in `MyDrive/voxtral-runs`,
so a disconnect loses at most a few minutes of training: reconnect, run cells 2, 3 and 4
again, and the script continues where it stopped.

## 1 · Check the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → L4 or A100"
name = torch.cuda.get_device_name(0)
memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}, {memory_gb:.0f} GB, bf16 supported: {torch.cuda.is_bf16_supported()}")
assert torch.cuda.is_bf16_supported() and memory_gb > 20, "Need an L4 or A100, not a T4"

## 2 · Mount Drive, get the code, install

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
url = "https://github.com/HFossdal/voxtral-norwegian.git"
if os.path.exists("/content/voxtral-norwegian"):
    !git -C /content/voxtral-norwegian pull -q {url} main
else:
    !git clone -q {url} /content/voxtral-norwegian
%cd /content/voxtral-norwegian
!git log --oneline -1
!pip install -q -e .
# Colab's preinstalled torchvision is built for its own torch version, not the
# pinned one, and can crash on import. This project doesn't use it.
!pip uninstall -y -q torchvision
# Same for torchao: peft refuses to add LoRA layers when an old torchao is
# installed (Colab ships 0.10; peft 0.21 needs >= 0.16). Not used here either.
!pip uninstall -y -q torchao
# pip then warns about other preinstalled Colab packages (cudf, diffusers, ...)
# wanting other versions of pandas, numba or pyarrow. They are not used here.

## 3 · Restart the runtime

The install replaces Colab's own torch and transformers with the project's pinned
versions. They are only used after a restart. This cell restarts the runtime on
purpose: Colab will report that the session crashed. **That is expected.** Then go on
with cell 4. You don't need to run cells 1–2 again.

The red `ERROR: pip's dependency resolver ...` lines above are warnings about
Colab packages this project never imports. The install itself succeeded if
it printed `Building editable for voxnor ... done`.


In [ ]:
import os

os.kill(os.getpid(), 9)

## 4 · Align, fine-tune, validate

This is the long step, in four parts that the log announces with `=== n/4 ... ===`:
1. download the training split (8 GB) and force-align its 18k transcripts;
2. fine-tune, scoring the model on 600 held-out validation clips at the start and every
   500 steps (lines `Selection set at step N: WER ...`);
3. score the best checkpoint on the full validation split;
4. write a report and a zip.

The test split is not used here.

In [ ]:
%cd /content/voxtral-norwegian
!python scripts/run_finetune.py --config configs/cloud.yaml --output-root /content/drive/MyDrive/voxtral-runs

## 5 · Look at the results

In [ ]:
import json
from pathlib import Path

from IPython.display import Markdown, display

root = Path("/content/drive/MyDrive/voxtral-runs")
run = sorted(root.glob("*_finetune-realtime"))[-1]
for line in (run / "selection.jsonl").read_text().splitlines():
    r = json.loads(line)
    print(f"step {r['step']:>5}: selection WER {100 * r['wer']:.2f}%")
print((run / "best.json").read_text())
print((run / "data_summary.json").read_text())
display(Markdown((root / "reports" / "finetune-realtime_validation.md").read_text()))

Everything worth keeping is also in `MyDrive/voxtral-runs/finetune-realtime_results.zip`.
Download it from Drive, or run the cell below.

In [ ]:
from google.colab import files

files.download("/content/drive/MyDrive/voxtral-runs/finetune-realtime_results.zip")